In [1]:
# Libraries
import numpy as np 
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform, transform_bounds # Reprojection
from rasterio.transform import Affine

In [2]:
# Only run initially!!

#from zipfile import ZipFile

#slope_zip = './data/slope/LF2020_SlpD_220_CONUS.zip'
#slope_out = './data/slope'

#with ZipFile(slope_zip, 'r') as zObject:
    # Extract downloaded slope data and store in data > slope folder
#    zObject.extractall(path = slope_out)

In [2]:
# Veiw raw data

slope_raw = './data/slope/LF2020_SlpD_220_CONUS/Tif/LC20_SlpD_220.tif'

with rasterio.open(slope_raw, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'int16', 'nodata': 32767.0, 'width': 156335, 'height': 101538, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2362395.0,
       0.0, -30.0, 3267405.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: 32767.0
CRS: EPSG:5070
Resolution (30.0, 30.0)
M

In [3]:
# Update profile 
    # dtype = int8
    # nodata = -10 (& replace nodata cells with new nodata value)

slope_raw = './data/slope/LF2020_SlpD_220_CONUS/Tif/LC20_SlpD_220.tif'
slope_profile_update = './data/slope/slope_profile_update.tif'

with rasterio.open(slope_raw) as src:
    profile = src.profile.copy()
    profile.update(dtype = rasterio.int8,
                   nodata = -10)

    with rasterio.open(slope_profile_update, 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window).astype(rasterio.int16) # int16 as intermediary 
            
            # Replace 32767.0 or -9999 (previous nodata values) to -10 (new nodata value)
            data[(data == 32767.0) | (data == -9999)] = -10
            
            # Write out new raster
            dst.write(data.astype(rasterio.int8), 1, window = window)

In [4]:
# Verify profile update

slope_profile_update = './data/slope/slope_profile_update.tif'

with rasterio.open(slope_profile_update, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 156335, 'height': 101538, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2362395.0,
       0.0, -30.0, 3267405.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resoluti

In [5]:
# Co-register slope with dist to gwt raster (inches convert)

# Define co-register function - BILINEAR RESAMPLING
def coregister_rasters(infile, match, outfile):
    """Reproject a file to match the shape and projection of existing raster. 
    
    Parameters
    ----------
    infile : (string) path to input file to reproject
    match : (string) path to raster with desired shape and projection 
    outfile : (string) path to output file tif
    """
    # Open input
    with rasterio.open(infile) as src:
        src_transform = src.transform
        
        # Open input to match
        with rasterio.open(match) as match:
            dst_crs = match.crs
            dst_transform = match.transform # Ensures resolutions of outfile and match will be exactly the same
            dst_width = match.width
            dst_height = match.height

        # Set properties for output
        dst_kwargs = src.meta.copy()
        dst_kwargs.update({'crs': dst_crs,
                           'transform': dst_transform,
                           'width': dst_width,
                           'height': dst_height,
                           'nodata': -10})
        print('Coregistered to shape:', dst_height, dst_width,'\n Affine', dst_transform)
        
        # Open output
        with rasterio.open(outfile, "w", **dst_kwargs) as dst:
            # Iterate through bands and write using reproject function
            for i in range(1, src.count + 1):
                reproject(
                    source = rasterio.band(src, i),
                    destination = rasterio.band(dst, i),
                    src_transform = src.transform,
                    src_crs = src.crs,
                    dst_transform = dst_transform,
                    dst_crs = dst_crs,
                    resampling = Resampling.bilinear)
                

# Apply coregister_rasters
slope_profile_update = './data/slope/slope_profile_update.tif' # Input 
ref_raster = './data/SSURGO_raw/dist_GWT/gwt_inches.tif' # Match
slope_coregistered = './data/slope/slope_coregistered.tif' # Output

coregister_rasters(
    infile = slope_profile_update,
    match = ref_raster,
    outfile = slope_coregistered
)

Coregistered to shape: 96751 153996 
 Affine | 30.00, 0.00,-2356125.00|
| 0.00,-30.00, 3172575.00|
| 0.00, 0.00, 1.00|


In [6]:
# Verify co-registered raster

slope_coregistered = './data/slope/slope_coregistered.tif' 

with rasterio.open(slope_coregistered, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 153996, 'blockysize': 1, 'tiled': False, 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolution (30.0, 30.0)
Mi

In [7]:
# Match with HSG final composite raster
hsg_final_composite = './data/SSURGO_raw/hsg/hsg_FINAL_composite.tif' 
slope_coregistered = './data/slope/slope_coregistered.tif' 
slope_MATCH = './data/slope/slope_MATCH.tif' 

# Open hsg composite raster - want conus cells to match THIS raster
with rasterio.open(hsg_final_composite) as conus:

    # Nodata value for the conus raster
    conus_nodata = conus.nodata 
    # Profile conus raster
    profile = conus.profile.copy()

    # Open raster - want to convert any cells containing data where conus contains NODATA to nodata
    with rasterio.open(slope_coregistered) as src:
        
        src_nodata = src.nodata # Nodata value 
        
        # Open output raster
        with rasterio.open(slope_MATCH, 'w', **profile) as dst:
        
            for ji, window in conus.block_windows(1):
            
                # hsg composite raster data
                conus_data = conus.read(1, window = window)
            
                # Land cover raster data
                src_data = src.read(1, window = window)
            
                # Identify cells where conus_data == nodata value
                remove_mask = (conus_data == conus_nodata)
            
                # Convert cells in src where conus is nodata to the nodata value
                src_data[remove_mask] = src_nodata
            
                # Write out
                dst.write(src_data, 1, window = window)

In [9]:
# Verify match raster

slope_MATCH = './data/slope/slope_MATCH.tif' 

with rasterio.open(slope_MATCH, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolutio

In [2]:
# Reclassify slope values to standard suitability scores
    # <= 20 - standard suitability score: 10
    # >20 - standard suitability score: 0
    
slope_MATCH = './data/slope/slope_MATCH.tif' 
slope_standardized = './data/slope/slope_standardized.tif' 
    
with rasterio.open(slope_MATCH, mode = 'r') as src:
    profile = src.profile.copy()
    profile.update(dtype = rasterio.float32)
    
    with rasterio.open(slope_standardized, 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window).astype(rasterio.float32)
            
            # Convert <= 20 to 10 (but >0 to NOT capture -10 nodata value)
            data[(data >=0) & (data <=20)] = 10
            
            # Convert >20 to 0
            data[data >20] = 0
            
            # Write out new raster
            dst.write(data, 1, window = window)

In [3]:
# Verify standardized values

slope_standardized = './data/slope/slope_standardized.tif' 

with rasterio.open(slope_standardized, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [4]:
# Verify standardized unique values 

slope_standardized = './data/slope/slope_standardized.tif' 

unique_values = set() # Create a set to store unique values

with rasterio.open(slope_standardized, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [ ]:
# Save out preprocessed percent impervious as cloud optimized GeoTiff - save to scratch

slope_MATCH = './data/slope/slope_MATCH.tif' 
cog_out_slope_raw_viewing = '/scratch/bfqp/cchan2/rasters/slope_RAW_VIEWING.tif' # SAVE TO SCRATCH

profile = cog_profiles.get('lzw')

profile.update(
    compress = 'DEFLATE',
    predictor = 3, # 2 for integer; 3 for float
    tiled = True,
    blockxsize = 512,
    blockysize = 512, 
    bigtiff = 'YES')

cog_translate(
    slope_MATCH, # dont include argument
    cog_out_slope_raw_viewing, # dont include argument
    profile, # dont include argument
    overview_resampling = 'bilinear', # Change based on data values
    nodata = -10,
    use_cog_driver = True,
    in_memory = False, 
    web_optimized = True)

Reading input: ./data/slope/slope_coregistered.tif

Adding overviews...
Updating dataset tags...
Writing output to: /scratch/bfqp/cchan2/rasters/slope_RAW_VIEWING.tif


In [8]:
# Validate cogeo raster
cog_out_slope_raw_viewing = '/scratch/bfqp/cchan2/rasters/slope_RAW_VIEWING.tif' 

cog_validate(cog_out_slope_raw_viewing)

(True, [], [])

In [9]:
# Copy from scratch back into projects folder
import shutil

cog_out_slope_raw_viewing = '/scratch/bfqp/cchan2/rasters/slope_RAW_VIEWING.tif' 
cog_out_slope_raw_viewing_PROJECTS = '/projects/bfqp/cchan2/data/slope/slope_RAW_VIEWING.tif'

shutil.copy(cog_out_slope_raw_viewing, cog_out_slope_raw_viewing_PROJECTS)

'/projects/bfqp/cchan2/data/slope/slope_RAW_VIEWING.tif'

In [10]:
# Verify processed cogeo raster

cog_out_impervious_raw_viewing_PROJECTS = './data/slope/slope_RAW_VIEWING.tif' 

with rasterio.open(cog_out_impervious_raw_viewing_PROJECTS, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 182784, 'height': 107776, 'count': 1, 'crs': CRS.from_wkt('PROJCS["WGS 84 / Pseudo-Mercator",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Mercator_1SP"],PARAMETER["central_meridian",0],PARAMETER["scale_factor",1],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],EXTENSION["PROJ4","+proj=merc +a=6378137 +b=6378137 +lat_ts=0 +lon_0=0 +x_0=0 +y_0=0 +k=1 +units=m +nadgrids=@null +wktext +no_defs"],AUTHORITY["EPSG","3857"]]'), 'transform': Affine(38.2185141425881, 0.0, -14245416.087451734,
       0.0, -38.2185141425881, 6731350.458905771), 'blockxsize': 512, 'blockysize': 512, 'tiled': True, 'compress': '

KeyboardInterrupt: 